# 02. ASR 평가, 오류 분석, 노이즈 강건성

음성 인식 모델을 개발할 때 핵심은 단순 데모가 아니라 `정규화된 평가`, `오류 유형`, `노이즈 조건별 성능`입니다.
이 노트북은 WER/CER을 직접 구현하고, 노이즈 조건별 오류표를 만듭니다.

선택 실습으로 공개 ASR 모델을 연결할 수 있지만, 기본 실습은 모델 다운로드 없이 진행됩니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
def edit_distance(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[-1][-1]

def normalize_text(text):
    import re
    text = text.lower().strip()
    text = re.sub(r"[^0-9a-z가-힣ぁ-んァ-ン一-龥\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def wer(ref, hyp):
    r = normalize_text(ref).split()
    h = normalize_text(hyp).split()
    return edit_distance(r, h) / max(1, len(r))

def cer(ref, hyp):
    r = list(normalize_text(ref).replace(" ", ""))
    h = list(normalize_text(hyp).replace(" ", ""))
    return edit_distance(r, h) / max(1, len(r))


In [ ]:
samples = [
    ("clean", "로봇이 왼쪽 선반의 빨간 상자를 집어 주세요", "로봇이 왼쪽 선반의 빨간 상자를 집어 주세요"),
    ("noise", "회의실 조명이 깜빡이고 있습니다", "회의실 조명이 깜박이고 있습니다"),
    ("noise", "請求書の合計金額を読んでください", "請求書の合計金額を呼んでください"),
    ("reverb", "start inspection after the alarm sound", "start inspection after alarm sounds"),
    ("overlap", "문서 번호 A17을 확인해 주세요", "문서 번호 a seventeen 확인해 주세요"),
    ("far_field", "환경음이 들리면 작업을 멈춰 주세요", "환경 음이 들리면 작업을 멈춰 주세요"),
]

eval_rows = []
for condition, ref, hyp in samples:
    eval_rows.append({
        "condition": condition,
        "reference": ref,
        "hypothesis": hyp,
        "wer": wer(ref, hyp),
        "cer": cer(ref, hyp),
    })

asr_eval = pd.DataFrame(eval_rows)
asr_eval.to_csv(ARTIFACTS / "asr_error_eval.csv", index=False, encoding="utf-8-sig")
asr_eval


In [ ]:
def classify_error(row):
    ref = normalize_text(row["reference"])
    hyp = normalize_text(row["hypothesis"])
    if ref == hyp:
        return "exact"
    if row["cer"] < 0.08:
        return "minor_spelling_or_spacing"
    if any(ch.isdigit() for ch in ref) or any(ch.isdigit() for ch in hyp):
        return "number_or_code"
    if row["wer"] > 0.4:
        return "severe_mismatch"
    return "semantic_or_token_error"

asr_eval["error_type"] = asr_eval.apply(classify_error, axis=1)
summary = asr_eval.groupby(["condition", "error_type"]).agg(
    n=("reference", "count"),
    mean_wer=("wer", "mean"),
    mean_cer=("cer", "mean"),
).reset_index()
summary.to_csv(ARTIFACTS / "asr_error_summary.csv", index=False, encoding="utf-8-sig")
summary


## 선택 실습: 실제 ASR 모델 연결

`transformers`, `torch`, `soundfile`을 설치한 뒤 공개 ASR 모델을 연결합니다.
모델 ID에 회사명 또는 조직명이 포함되어 보고서에 노출될 수 있으면 본문에서는 `xxx/model-name`처럼 마스킹하고,
재현용 설정 파일에는 별도 비공개 변수로 둡니다.

```python
# from transformers import pipeline
# asr = pipeline("automatic-speech-recognition", model="모델_ID")
# result = asr("sample.wav")
# print(result["text"])
```

논문에서는 모델명 자체보다 `모델 크기`, `학습 언어`, `추론 속도`, `WER/CER`을 중심으로 비교합니다.
